In [ ]:
!pip install -U sentence-transformers git+https://github.com/huggingface/transformers@v4.56.0-Embedding-Gemma-preview

  Cloning https://github.com/huggingface/transformers (to revision v4.56.0-Embedding-Gemma-preview) to /tmp/pip-req-build-bo7uevd1
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-bo7uevd1
  Running command git checkout -q 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Resolved https://github.com/huggingface/transformers to commit 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.8 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-4.57.0.dev0-py3-none-any.whl size=12604658 sha256=f8c234fd16d3cbe5b6ed87d21d66584dc20c0a778d15af5f369a754ba8dd9978
  Stored in directory: /tmp/pip-ephem-wheel-cache-l6s_gy5v/wheels/3a/21/76/c31899bac2cf601d3c74091b26a413bc3fb54770d5ccb5c924
Successfully built transformers
  Attempting uni

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
WANDB_TOKEN = userdata.get('WANDB_KEY')

In [ ]:
# Login into Hugging Face Hub
from huggingface_hub import login
login(token=HF_TOKEN)

import wandb
wandb.login(key=WANDB_TOKEN)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score
import wandb
from datasets import load_dataset
from tqdm import tqdm
import numpy as np

In [ ]:
ds_train = load_dataset("dinushiTJ/nz_research_commons_embedding_triplets_5k", split="train")
ds_test = load_dataset("dinushiTJ/nz_research_commons_embedding_triplets_5k", split="test")

def rename_cols(example):
    return {
        "anchor": example["label"],
        "positive": example["positive"],
        "negative": example["negative"],
    }

trainset = ds_train.map(rename_cols, remove_columns=["label"])
testset = ds_test.map(rename_cols, remove_columns=["label"])

README.md:   0%|          | 0.00/487 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/9.27M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3318 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3318 [00:00<?, ? examples/s]

In [ ]:
trainset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 5000
})

In [ ]:
testset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 3318
})

In [ ]:
trainset[0]

{'anchor': 'maori_origin',
 'positive': 'title: KAUMĀTUATANGA  Supporting School Leaders To Develop Cultural Values While Resisting The Dominance of Colonialism\n\nauthors: Nuri, Ngahuia\n\nsubjects: Kaumātua\n\nabstract: In an education system that is asserting to the importance of Māori (indigenous people of Aotearoa) language, culture, identity and the history of Aotearoa  (New Zealand) into the curriculum, there is direction from the Ministry of Education (MOE) for schools to reach out to iwi. What this looks like and how this can be achieved is not an easy task. This thesis follows the journey of three kaumātua (respected, knowledgeable elders, both female and male) working alongside the Senior Leadership Team (SLT) of a decile 10 kura auraki (mainstream primary school) where ākonga Māori (Māori students) were in the minority. The research examines what role kaumātua might have in guiding other schools to help tamariki Māori (Māori children) enjoy and achieve education success as 

In [ ]:
task_name = "Classification"

CONTEXT_LEN = 2048  # Gemma context length
EMBED_DIM = 768     # Gemma embedding size
BATCH_SIZE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

model_id = "google/embeddinggemma-300M"
model = SentenceTransformer(model_id).to(device=DEVICE)

print(f"Device: {model.device}")
print(model)
print("Total number of parameters in the model:", sum([p.numel() for _, p in model.named_parameters()]))

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

Device: cuda:0
SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'Gemma3TextModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 3072, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity', 'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
  (3): Dense({'in_features': 3072, 'out_features': 768, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity', 'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
  (4): Normalize({})
)
Total number of parameters in the model: 307581696


In [ ]:
def get_doc_embedding(doc: str):
    """
    If doc tokens > CONTEXT_LEN, split into chunks, embed each, and average
    """
    tokens = doc.split()  # simple tokenization, you can use HF tokenizer if needed
    if len(tokens) <= CONTEXT_LEN:
        emb = model.encode(doc, prompt_name=task_name)
        return torch.tensor(emb, dtype=torch.float32)
    else:
        # chunking
        chunk_embs = []
        for i in range(0, len(tokens), CONTEXT_LEN):
            chunk = " ".join(tokens[i:i+CONTEXT_LEN])
            chunk_emb = model.encode(chunk, prompt_name=task_name)
            chunk_embs.append(torch.tensor(chunk_emb, dtype=torch.float32))
        # mean pooling
        return torch.stack(chunk_embs, dim=0).mean(dim=0)


In [ ]:
def get_scores(query, documents):
  # Calculate embeddings by calling model.encode()
  query_embeddings = model.encode(query, prompt_name=task_name)
  doc_embeddings = model.encode([d.get("doc") for d in documents], prompt_name=task_name)

  # Calculate the embedding similarities
  similarities = model.similarity(query_embeddings, doc_embeddings)

  for idx, doc in enumerate(documents):
    print("Document: ", doc.get("d"), "-> 🤖 Score: ", similarities.numpy()[0][idx], f'[{doc.get("doc")[:30]}...]')

query = testset[0]["anchor"]
print(query)
documents = [{"doc": testset[0]["positive"], "d": "positive"}, {"doc": testset[0]["negative"], "d": "negative"}]

get_scores(query, documents)

maori_origin
Document:  positive -> 🤖 Score:  0.5779943 [title: Lessons from children i...]
Document:  negative -> 🤖 Score:  0.43200853 [title: Developing a motor anal...]


## Training

Using a framework like `sentence-transformers` in Python, the base model gradually learns the subtle distinctions in your financial vocabulary.

In [ ]:
assert task_name in model.prompts

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from transformers import TrainerCallback
from sentence_transformers.sentence_transformer.losses import TripletDistanceMetric
from sentence_transformers.sentence_transformer.evaluation import TripletEvaluator

In [ ]:
from sentence_transformers import SentenceTransformerTrainingArguments

args = SentenceTransformerTrainingArguments(
    output_dir="rc-embedding-gemma",

    prompts=model.prompts[task_name],

    num_train_epochs=1,
    per_device_train_batch_size=1,
    learning_rate=2e-5,
    warmup_ratio=0.1,

    logging_steps=50,
    logging_strategy="steps",

    eval_steps=100,
    eval_strategy="steps",

    save_steps=100,
    save_strategy="steps",

    save_total_limit=2,

    report_to="wandb",
    run_name="rc-embedding-gemma-300m",

    load_best_model_at_end=True,
    metric_for_best_model="eval_rc-triplet-eval_cosine_accuracy",
    greater_is_better=True,

    eval_on_start=True,

    # NEW
    hub_model_id="dinushiTJ/nz-research-commons-embedding-gemma",
    hub_strategy="checkpoint",
    hub_private_repo=False,
    push_to_hub=True,
)

In [ ]:
# from sentence_transformers import SentenceTransformer

# model = SentenceTransformer(
#     "rc-embedding-gemma/checkpoint-1800"
# ).to(device=DEVICE)

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "dinushiTJ/nz-research-commons-embedding-gemma"
)

for f in files:
    if "last-checkpoint" in f:
        print(f)

last-checkpoint/1_Pooling/config.json
last-checkpoint/2_Dense/config.json
last-checkpoint/2_Dense/model.safetensors
last-checkpoint/3_Dense/config.json
last-checkpoint/3_Dense/model.safetensors
last-checkpoint/README.md
last-checkpoint/added_tokens.json
last-checkpoint/config.json
last-checkpoint/config_sentence_transformers.json
last-checkpoint/model.safetensors
last-checkpoint/modules.json
last-checkpoint/optimizer.pt
last-checkpoint/rng_state.pth
last-checkpoint/scheduler.pt
last-checkpoint/sentence_bert_config.json
last-checkpoint/special_tokens_map.json
last-checkpoint/tokenizer.json
last-checkpoint/tokenizer.model
last-checkpoint/tokenizer_config.json
last-checkpoint/trainer_state.json
last-checkpoint/training_args.bin


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="dinushiTJ/nz-research-commons-embedding-gemma",
    local_dir="hf_checkpoint"
)

Fetching 40 files:   0%|          | 0/40 [00:00<?, ?it/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

training_args.bin:   0%|          | 0.00/6.22k [00:00<?, ?B/s]

'/content/hf_checkpoint'

In [ ]:
import os

print(os.listdir("hf_checkpoint"))
print(os.listdir("hf_checkpoint/last-checkpoint"))

['2_Dense', 'added_tokens.json', 'last-checkpoint', 'sentence_bert_config.json', 'tokenizer.model', 'config_sentence_transformers.json', 'special_tokens_map.json', '.cache', 'training_args.bin', 'README.md', 'eval', 'config.json', 'modules.json', '3_Dense', '1_Pooling', 'tokenizer_config.json', '.gitattributes', 'tokenizer.json', 'model.safetensors']
['2_Dense', 'rng_state.pth', 'added_tokens.json', 'trainer_state.json', 'sentence_bert_config.json', 'scheduler.pt', 'tokenizer.model', 'config_sentence_transformers.json', 'special_tokens_map.json', 'training_args.bin', 'README.md', 'config.json', 'modules.json', '3_Dense', 'optimizer.pt', '1_Pooling', 'tokenizer_config.json', 'tokenizer.json', 'model.safetensors']


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "hf_checkpoint/last-checkpoint"
).to(device=DEVICE)

In [ ]:
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.sentence_transformer.losses import TripletLoss, TripletDistanceMetric
from sentence_transformers.sentence_transformer.evaluation import TripletEvaluator

# The model is already defined and available from a previous cell (QohTTml3JFiQ).
# model = SentenceTransformer(model_id).to(device=DEVICE)

# Define loss and evaluator
loss = TripletLoss(model, distance_metric=TripletDistanceMetric.COSINE, triplet_margin=0.3);

evaluator = TripletEvaluator(
    anchors=testset["anchor"],
    positives=testset["positive"],
    negatives=testset["negative"],
    name="rc-triplet-eval"
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=trainset,
    loss=loss,
    evaluator=evaluator,
)

trainer.train(
    resume_from_checkpoint="hf_checkpoint/last-checkpoint"
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Rc-triplet-eval Cosine Accuracy
5000,0.013600,No log,0.984027


ERROR:sentence_transformers.base.trainer:Could not load the best model from rc-embedding-gemma/checkpoint-3800. Error: 404 Client Error. (Request ID: Root=1-6a1fc8f7-53aa151b4ee8577b34039e39;23122023-4a93-44dd-9f76-4079f26db64e)

Repository Not Found for url: https://huggingface.co/rc-embedding-gemma/checkpoint-3800/resolve/main/modules.json.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication


TrainOutput(global_step=5000, training_loss=0.0, metrics={'train_runtime': 758.7803, 'train_samples_per_second': 6.59, 'train_steps_per_second': 6.59, 'total_flos': 0.0, 'train_loss': 0.0, 'epoch': 1.0})

In [ ]:
from huggingface_hub import hf_hub_download
import json

path = hf_hub_download(
    repo_id="dinushiTJ/nz-research-commons-embedding-gemma",
    filename="last-checkpoint/trainer_state.json"
)

with open(path) as f:
    state = json.load(f)

print("global_step =", state["global_step"])
print("best_global_step =", state["best_global_step"])

trainer_state.json: 0.00B [00:00, ?B/s]

global_step = 5000
best_global_step = 3800


In [ ]:
print("loss" in globals())
print("evaluator" in globals())
print("trainset" in globals())
print("args" in globals())

True
True
True
True


In [ ]:
# Print the absolute directory path of the best checkpoint
print("Best checkpoint path:", trainer.state.best_model_checkpoint)

# Print the corresponding best metric score achieved at that checkpoint
print("Best metric value:", trainer.state.best_metric)


Best checkpoint path: rc-embedding-gemma/checkpoint-3800
Best metric value: 0.9870404005050659


In [ ]:
# Check the configuration path history
print(trainer.model.config._name_or_path)


hf_checkpoint/last-checkpoint


In [ ]:
trainer.save_model("rc-embedding-gemma-final")
trainer.save_state()

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ding-gemma/tokenizer.json:  72%|#######1  | 23.9MB / 33.4MB            

  ...ing-gemma/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...g-gemma/model.safetensors:   2%|2         | 28.4MB / 1.21GB            

  ...2_Dense/model.safetensors: 100%|##########| 9.44MB / 9.44MB            

  ...3_Dense/model.safetensors: 100%|##########| 9.44MB / 9.44MB            

  ...g-gemma/training_args.bin:   6%|6         |   375B / 6.22kB            

In [ ]:
trainer.push_to_hub("Upload best model based on cosine accuracy")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ding-gemma/tokenizer.json:  24%|##3       | 7.98MB / 33.4MB            

  ...g-gemma/training_args.bin: 100%|##########| 6.22kB / 6.22kB            

  ...g-gemma/model.safetensors:   1%|          | 7.99MB / 1.21GB            

  ...ing-gemma/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...3_Dense/model.safetensors:  84%|########4 | 7.97MB / 9.44MB            

  ...2_Dense/model.safetensors:  85%|########4 | 8.00MB / 9.44MB            

CommitInfo(commit_url='https://huggingface.co/dinushiTJ/nz-research-commons-embedding-gemma/commit/a4183de020bd899940291b7505fcc109923908ff', commit_message='Upload best model based on cosine accuracy', commit_description='', oid='a4183de020bd899940291b7505fcc109923908ff', pr_url=None, repo_url=RepoUrl('https://huggingface.co/dinushiTJ/nz-research-commons-embedding-gemma', endpoint='https://huggingface.co', repo_type='model', repo_id='dinushiTJ/nz-research-commons-embedding-gemma'), pr_revision=None, pr_num=None)

In [ ]:
# hub_strategy="checkpoint"
# push_to_hub=True
# hub_model_id="dinushiTJ/nz-research-commons-embedding-gemma"

In [ ]:
print(args.push_to_hub)
print(args.hub_strategy)
print(args.save_total_limit)


True
HubStrategy.CHECKPOINT
2


In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files("dinushiTJ/nz-research-commons-embedding-gemma")

for f in files[:50]:
    print(f)

.gitattributes
1_Pooling/config.json
2_Dense/config.json
2_Dense/model.safetensors
3_Dense/config.json
3_Dense/model.safetensors
README.md
added_tokens.json
config.json
config_sentence_transformers.json
eval/triplet_evaluation_rc-triplet-eval_results.csv
last-checkpoint/1_Pooling/config.json
last-checkpoint/2_Dense/config.json
last-checkpoint/2_Dense/model.safetensors
last-checkpoint/3_Dense/config.json
last-checkpoint/3_Dense/model.safetensors
last-checkpoint/README.md
last-checkpoint/added_tokens.json
last-checkpoint/config.json
last-checkpoint/config_sentence_transformers.json
last-checkpoint/model.safetensors
last-checkpoint/modules.json
last-checkpoint/optimizer.pt
last-checkpoint/rng_state.pth
last-checkpoint/scheduler.pt
last-checkpoint/sentence_bert_config.json
last-checkpoint/special_tokens_map.json
last-checkpoint/tokenizer.json
last-checkpoint/tokenizer.model
last-checkpoint/tokenizer_config.json
last-checkpoint/trainer_state.json
last-checkpoint/training_args.bin
model.safe

In [ ]:
# from huggingface_hub import list_repo_files

# files = list_repo_files("dinushiTJ/nz-research-commons-embedding-gemma")
# print([f for f in files if "last-checkpoint" in f])

### After GPU Issue,
- deleted corrupt `1900-checkpoint` and till `1200-checkpoint`
- `12 × 3.5 GB ≈ 42 GB available`
- Start from where ended.

In [ ]:
print(args.save_total_limit)
print(args.save_steps)

2
100


In [ ]:
# import json

# with open("rc-embedding-gemma/checkpoint-1800/trainer_state.json") as f:
#     state = json.load(f)

# print(state["global_step"])

## After Fine-Tuning

The same search now yields much clearer results:

1. Document: Opening a NISA account (Score: 0.73) <- *Much more confident*
2. Document: Opening a Regular Saving Account (Score: 0.34) <- *Clearly less relevant*
3. Document: Home Loan Application Guide (Score: 0.47)

In [ ]:
get_scores(query, documents)

Document:  positive -> 🤖 Score:  0.77506286 [title: Lessons from children i...]
Document:  negative -> 🤖 Score:  -0.8222812 [title: Developing a motor anal...]


In [ ]:
# # Push to Hub
# model_id = "dinushiTJ/nz-research-commons-embedding-gemma"
# model.push_to_hub(model_id) -- no need, already saved

## Evaluate Base Model Performance

### Subtask:
Load the base 'google/embeddinggemma-300M' model and evaluate its performance on the 'hf_testset'. This involves calculating the average similarity between anchor and positive examples, and anchor and negative examples, and determining the average gap between these similarities.


In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the base model 'google/embeddinggemma-300M'
base_model_id = "google/embeddinggemma-300M"
base_model = SentenceTransformer(base_model_id).to(device=device)

print(f"Base Model Device: {base_model.device}")

# Define the task_name for the base model's prompt
task_name_base = "Classification"

# Extract texts from hf_testset
anchors = testset["anchor"]
positives = testset["positive"]
negatives = testset["negative"]

# Encode all texts using the base model
anchor_embeddings_base = base_model.encode(anchors, prompt_name=task_name_base, convert_to_tensor=True, normalize_embeddings=True)
positive_embeddings_base = base_model.encode(positives, prompt_name=task_name_base, convert_to_tensor=True, normalize_embeddings=True)
negative_embeddings_base = base_model.encode(negatives, prompt_name=task_name_base, convert_to_tensor=True, normalize_embeddings=True)

# Calculate cosine similarity
ap_similarities = torch.nn.functional.cosine_similarity(anchor_embeddings_base, positive_embeddings_base)
an_similarities = torch.nn.functional.cosine_similarity(anchor_embeddings_base, negative_embeddings_base)

# Calculate averages
average_ap_similarity = torch.mean(ap_similarities).item()
average_an_similarity = torch.mean(an_similarities).item()

# Calculate the average gap
average_gap_base = average_ap_similarity - average_an_similarity

# Print the results
print(f"\nBase Model Performance on hf_testset:")
print(f"Average Anchor-Positive Similarity (Base Model): {average_ap_similarity:.4f}")
print(f"Average Anchor-Negative Similarity (Base Model): {average_an_similarity:.4f}")
print(f"Average Gap (Anchor-Positive - Anchor-Negative) (Base Model): {average_gap_base:.4f}")

Base Model Device: cuda:0

Base Model Performance on hf_testset:
Average Anchor-Positive Similarity (Base Model): 0.4670
Average Anchor-Negative Similarity (Base Model): 0.5547
Average Gap (Anchor-Positive - Anchor-Negative) (Base Model): -0.0877


## Evaluate Fine-Tuned Model Performance

### Subtask:
Evaluate the already fine-tuned model on the 'hf_testset'. This will involve calculating and printing the average similarity between anchor and positive examples, anchor and negative examples, and the average gap between these similarities, similar to the base model evaluation.


In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the ft model
model_id = "dinushiTJ/nz-research-commons-embedding-gemma"
ft_model = SentenceTransformer(model_id).to(device=device)

print(f"FT Model Device: {ft_model.device}")

# Define the task_name for the ft model's prompt
task_name_ft = "Classification"

# Encode all texts using the fine-tuned model
anchor_embeddings_ft = ft_model.encode(anchors, prompt_name=task_name_ft, convert_to_tensor=True, normalize_embeddings=True)
positive_embeddings_ft = ft_model.encode(positives, prompt_name=task_name_ft, convert_to_tensor=True, normalize_embeddings=True)
negative_embeddings_ft = ft_model.encode(negatives, prompt_name=task_name_ft, convert_to_tensor=True, normalize_embeddings=True)

# Calculate cosine similarity for the fine-tuned model
ap_similarities_ft = torch.nn.functional.cosine_similarity(anchor_embeddings_ft, positive_embeddings_ft)
an_similarities_ft = torch.nn.functional.cosine_similarity(anchor_embeddings_ft, negative_embeddings_ft)

# Calculate averages for the fine-tuned model
average_ap_similarity_ft = torch.mean(ap_similarities_ft).item()
average_an_similarity_ft = torch.mean(an_similarities_ft).item()

# Calculate the average gap for the fine-tuned model
average_gap_ft = average_ap_similarity_ft - average_an_similarity_ft

# Print the results for the fine-tuned model
print(f"\nFine-Tuned Model Performance on hf_testset:")
print(f"Average Anchor-Positive Similarity (Fine-Tuned Model): {average_ap_similarity_ft:.4f}")
print(f"Average Anchor-Negative Similarity (Fine-Tuned Model): {average_an_similarity_ft:.4f}")
print(f"Average Gap (Anchor-Positive - Anchor-Negative) (Fine-Tuned Model): {average_gap_ft:.4f}")


modules.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

FT Model Device: cuda:0

Fine-Tuned Model Performance on hf_testset:
Average Anchor-Positive Similarity (Fine-Tuned Model): 0.7375
Average Anchor-Negative Similarity (Fine-Tuned Model): -0.6331
Average Gap (Anchor-Positive - Anchor-Negative) (Fine-Tuned Model): 1.3706


End